# SmolLM2 Limbic System Training

Fine-tunes **SmolLM2-135M-Instruct** with LoRA to function as an emotional processing layer.

Three input types:
- **Sensory:** 6D neural activation vector + context → emotions + attention + drives
- **Thought:** Inner monologue text (GoEmotions dataset) → emotions + attention + drives
- **Integrated:** Both sensory + thought → blended emotional state

**Recommended runtime:** T4 GPU (free tier) — ~20 min training.

**VRAM:** ~2GB peak (4-bit quantized 135M model + LoRA)

In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes scipy

In [ ]:
# Upload limbic_training.py to Colab, or clone repo
# from google.colab import files
# files.upload()  # select limbic_training.py

import limbic_training as lt

## 1. Generate Training Data

- 5,000 sensory examples (6D vector → emotions)
- 10,000 thought examples (GoEmotions dataset: `google-research-datasets/go_emotions`)
- 10,000 integrated examples (vector + text → blended emotions)

In [ ]:
train_ds, eval_ds = lt.build_dataset(use_goemotions=True)
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")
print(f"\nSample:\n{train_ds[0]['text'][:500]}")

## 2. Inspect Data Distribution

In [ ]:
sensory_count = sum(1 for ex in train_ds if 'Neural Activation' in ex['text'] and 'Inner Thought' not in ex['text'])
thought_count = sum(1 for ex in train_ds if 'Inner Thought' in ex['text'] and 'Neural Activation' not in ex['text'])
integrated_count = sum(1 for ex in train_ds if 'Neural Activation' in ex['text'] and 'Inner Thought' in ex['text'])
print(f"Sensory-only: {sensory_count}")
print(f"Thought-only: {thought_count}")
print(f"Integrated:   {integrated_count}")

## 3. Train

SmolLM2-135M-Instruct with LoRA (rank 8, q_proj + v_proj). 4-bit quantized.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

model_id = lt.BASE_MODEL_ID
output_dir = lt.DEFAULT_OUTPUT_DIR

# Auto-detect bf16 support (A100/H100 = bf16, T4/L4 = fp16)
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Using {'bf16' if use_bf16 else 'fp16'} on {torch.cuda.get_device_name()}")
print(f"Base model: {model_id}")

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    max_length=512,
)

print("Starting training...")
trainer.train()

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

## 4. Evaluate

In [ ]:
lt.evaluate(model, tokenizer)

## 5. Interactive Demo

In [ ]:
# Sensory input
response = lt.infer(model, tokenizer,
    lt.format_sensory_input([0.10, 0.85, 0.10, 0.50, 0.75, 0.40], "sudden unexpected stimulus")
)
print(response)

In [ ]:
# Thought input
response = lt.infer(model, tokenizer,
    lt.format_thought_input("What is that strange light? I've never seen anything like it.")
)
print(response)

In [ ]:
# Integrated: conflict pattern
response = lt.infer(model, tokenizer,
    lt.format_integrated_input(
        [0.65, 0.65, 0.50, 0.45, 0.75, 0.20],
        "novel object in restricted area",
        "This looks fascinating but I really shouldn't be here"
    )
)
print(response)

In [ ]:
# Integrated: reappraisal pattern
response = lt.infer(model, tokenizer,
    lt.format_integrated_input(
        [0.10, 0.85, 0.10, 0.30, 0.75, 0.15],
        "loud noise at night",
        "Oh wait, that's just the neighbor's cat knocking things over again"
    )
)
print(response)

## 6. Export Merged Model (for deployment)

Merges LoRA weights into the base SmolLM2-135M model. Output is a self-contained model (~270MB fp16).

In [ ]:
lt.merge_and_export(output_dir)

In [ ]:
# Download the LoRA adapter (~5MB) or merged model (~270MB)
from google.colab import files

# LoRA adapter only (recommended — small, loads on top of base model)
!zip -r limbic-lora.zip ./limbic-smollm2-lora/
files.download('limbic-lora.zip')

# Merged model (self-contained, larger)
# !zip -r limbic-merged.zip ./limbic-smollm2-lora-merged/
# files.download('limbic-merged.zip')